# Continue fine-tuning Grounding DINO: DIOR-RSVG + VRSBench + DOTA

**This is v3**, replacing the in-progress `kaggle_finetune_grounding_dino_v2_continued.ipynb` run
(which added VRSBench on top of DIOR-RSVG). That run was cancelled by Kaggle
(`KernelWorkerStatus.CANCEL_ACKNOWLEDGED`) after running long enough to consume ~15 of the
account's 30 weekly GPU-hours -- almost certainly Kaggle's own session time limit, not a training
failure. Rather than dig through its output to see how far it got, this is a clean restart with a
smaller, safer epoch/data budget so the same thing doesn't happen twice.

**Same base as v1** (`kaggle_finetune_grounding_dino_dior_rsvg.ipynb`, which trained the
`dior_rsvg_finetuned.pth` checkpoint currently in `models/grounding/checkpoints/`) -- same setup,
same DIOR-RSVG parsing, same eval protocol. Three things layered on top:

1. **Resumes from the current checkpoint** (`PRETRAIN_MODEL_PATH`, not `--resume` -- loads weights
   and starts a fresh training schedule rather than restoring optimizer/epoch state).
2. **VRSBench's `[refer]` examples** (same as v2 -- capped at `MAX_VRSBENCH_IMAGES`, with the
   DIOR-RSVG-test-split leak filtered out, verified against the real data in v2).
3. **DOTA** (new): a fixed-category aerial object-detection dataset (15 categories in v1.0, +1 in
   v1.5 -- `plane`, `ship`, `storage-tank`, `harbor`, `bridge`, `helicopter`, etc.), added via
   Open-GroundingDino's *detection*-style ODVG format (fixed `label_map`, not free-text captions)
   rather than the grounding-style format DIOR-RSVG/VRSBench use -- verified against
   Open-GroundingDino's own `data_format.md` before writing the conversion cell below, not
   guessed. Sourced from the Kaggle-hosted mirror `kolos1/dota-coco-format` (Add Input, no local
   download): 512x512 patches (DOTA's native images run up to 20,000x20,000px and need splitting
   before any detector can use them -- this mirror already did that) with real COCO-format
   (axis-aligned) annotations -- verified directly by downloading and parsing its
   `instances_val.json` before committing to it (2000 images, 29024 boxes, 16 sane category
   names), not just trusting the file listing. **License note**: the Kaggle mirror's own page tags
   this MIT, but DOTA's actual upstream license (captain-whu.github.io/DOTA) is academic/
   non-commercial use only, same as DIOR-RSVG's CC BY-NC 4.0 -- that's the real constraint that
   applies here regardless of the mirror's tag.

**Data budget, sized to avoid repeating v2's cancellation**: DIOR-RSVG's 26,991 training examples
alone is already most of a safe per-epoch step budget on a T4 at `batch_size=6`. VRSBench and DOTA
are both capped at a few thousand images each (`MAX_VRSBENCH_IMAGES` / `MAX_DOTA_IMAGES` below) --
real added supervision without multiplying the dataset several times over -- and `epochs` is 2
(down from v2's 3), with a checkpoint saved after **every** epoch (`save_checkpoint_interval=1`,
down from v2's 3) so a session that still gets cut early leaves something usable behind, unlike
v2's run.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU** (T4x2 or P100) in the notebook's
Settings panel before starting.

In [ ]:
import torch, subprocess
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here. Landing on a P100 broke the custom CUDA-op build in Section 1 with `CUDA error: no kernel
image is available for execution on the device` -- a real run, not a hypothetical. The cell below
detects the actual GPU via `nvidia-smi` (before torch is ever imported, so there's no stale-module
issue) and reinstalls a CUDA 11.8 build first if needed -- those wheels cover Pascal through
Hopper, so this works regardless of which GPU Kaggle assigns.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

## 1. Setup — clone Open-GroundingDino, install deps, build the CUDA ops

If the CUDA-op build fails, it's almost always a CUDA/PyTorch version mismatch. Common fix: check
`nvcc --version` vs `torch.version.cuda` and, if they disagree, either install a matching `nvcc`
via `conda install -c nvidia cuda-nvcc=<version>` or pin `torch` to match the system CUDA before
re-running the build.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/longzw1997/Open-GroundingDino.git
%cd Open-GroundingDino
!pip install -q -r requirements.txt
# Open-GroundingDino calls BertModel.get_head_mask, which transformers 5.x removed -- the
# Kaggle image now ships 5.x, so training dies building the model without this pin.
!pip install -q "transformers<5"
!pip install -q gdown pycocotools

### PyTorch 2.6+ `torch.load` default change

**Found live**: got past every previous blocker this time, ~16 minutes into distributed training,
and died loading the continue-from checkpoint: `_pickle.UnpicklingError: ... Unsupported global:
GLOBAL argparse.Namespace was not an allowed global by default`. Root cause, confirmed by reading
`main.py` directly: Open-GroundingDino's checkpoints bundle an `argparse.Namespace` alongside the
weights, and PyTorch 2.6+ flipped `torch.load`'s default from `weights_only=False` to `True` --
breaking any older codebase's checkpoint loading that doesn't pass that argument explicitly, which
`main.py` doesn't. Patched below rather than edited by hand, since this repo is freshly cloned
every run -- all three `torch.load(...)` calls in `main.py`, not just the one that fires for this
notebook's specific args, in case a future edit changes which path gets used. Safe here since
these are all our own or the official release's checkpoints, not an untrusted download.

In [ ]:
import re

main_py = "/kaggle/working/Open-GroundingDino/main.py"
with open(main_py) as f:
    content = f.read()
patched = re.sub(
    r"torch\.load\(([^)]+?)\)",
    lambda m: m.group(0) if "weights_only" in m.group(1) else f"torch.load({m.group(1)}, weights_only=False)",
    content,
)
n_patched = len(re.findall(r"weights_only=False", patched))
with open(main_py, "w") as f:
    f.write(patched)
print(f"Patched main.py: {n_patched} torch.load(...) call(s) now pass weights_only=False")
assert n_patched == 3, f"expected 3 torch.load(...) calls in main.py, patched {n_patched} -- upstream file may have changed, check manually"

In [ ]:
%cd /kaggle/working/Open-GroundingDino/models/GroundingDINO/ops
!python setup.py build install
!python test.py   # should print a bunch of "True" — confirms the compiled op matches the pure-pytorch fallback
%cd /kaggle/working/Open-GroundingDino

## 2. Download pretrained weights (Swin-T checkpoint + warm the BERT cache)

In [ ]:
%cd /kaggle/working/Open-GroundingDino
!mkdir -p weights
!wget -q -P weights https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
print("Downloaded:", __import__("os").path.getsize("weights/groundingdino_swint_ogc.pth"), "bytes")

# bert-base-uncased auto-downloads from the HF hub the first time the model is built (public, no
# token needed) -- warm the cache here so training doesn't stall on it mid-run.
from transformers import AutoTokenizer, AutoModel
AutoTokenizer.from_pretrained("bert-base-uncased")
AutoModel.from_pretrained("bert-base-uncased")
print("BERT cached.")

## 2b. Upload your CURRENT fine-tuned checkpoint

Before running this cell: on the right-hand panel of the Kaggle notebook editor, click
**Add Input -> Upload -> New Dataset**, upload your local
`models/grounding/checkpoints/dior_rsvg_finetuned.pth`, and name the dataset something like
`dior-rsvg-finetuned-v1`. It'll then be mounted read-only under
`/kaggle/input/dior-rsvg-finetuned-v1/dior_rsvg_finetuned.pth` (Kaggle lowercases/hyphenates the
dataset name for the path -- check the actual path in the right-hand panel after upload and adjust
`CURRENT_CKPT_GLOB` below if it doesn't match).

**Also add `kolos1/dota-coco-format` as a second input here** (Add Input -> search "dota-coco-format")
-- Section 4b below reads it from `/kaggle/input/`.

In [ ]:
import glob, os

# Recursive ("**"), not a single "*" level -- confirmed live that a kernel pushed via the
# Kaggle API mounts dataset_sources one level deeper (/kaggle/input/datasets/<owner>/<slug>/...)
# than the classic web-UI "Add Input" flow (/kaggle/input/<slug>/...) this glob was written for.
CURRENT_CKPT_GLOB = "/kaggle/input/**/dior_rsvg_finetuned.pth"
matches = glob.glob(CURRENT_CKPT_GLOB, recursive=True)
if not matches:
    # Self-diagnosing on failure -- an earlier API-driven run failed this exact assertion with no
    # indication of what (if anything) actually mounted under /kaggle/input, so print that first.
    print("Contents of /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(" ", os.path.join(root, f))
    if not os.path.isdir("/kaggle/input") or not os.listdir("/kaggle/input"):
        print("  (nothing mounted under /kaggle/input at all)")
assert matches, (
    f"No checkpoint found matching {CURRENT_CKPT_GLOB} -- did you upload it as a Kaggle input "
    "dataset yet? See the markdown cell above. (If this notebook was pushed via the Kaggle API "
    "with dataset_sources set and it's still not here, check the printed /kaggle/input listing "
    "above for the actual mount path/name and adjust CURRENT_CKPT_GLOB.)"
)
CURRENT_CKPT = matches[0]
print("Using current checkpoint:", CURRENT_CKPT)

## 3. Download DIOR-RSVG

Canonical source: the DIOR-RSVG authors' Google Drive folder (linked from
https://github.com/ZhanYang-nwpu/RSVG-pytorch). `gdown --folder` pulls the whole folder; if it
stalls or hits Google's "too many downloads" warning, re-run the cell (gdown resumes) or grab the
zip manually and upload it as a Kaggle Dataset instead — either way you want the layout below.

Expected layout:
```
DIOR_RSVG/
  Annotations/   *.xml   (bbox + referring expression per object)
  JPEGImages/    *.jpg
  train.txt      (26991 object-level indices)
  val.txt        (3829)
  test.txt       (7500)
```

In [ ]:
%cd /kaggle/working
!gdown --folder "https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_" -O DIOR_RSVG
!echo "---"
!find DIOR_RSVG -maxdepth 2 | head -20

# gdown pulls the folder's contents as-is -- sometimes that's already-extracted files, sometimes
# it's zip archives (Annotations.zip / JPEGImages.zip) that still need unzipping. Either way, any
# zip found gets extracted then DELETED immediately -- keeping both the zip and its extracted
# contents on disk at once was blowing past Kaggle's working-directory quota on a live run.
import glob, os
for zip_path in glob.glob("DIOR_RSVG/*.zip"):
    print(f"Extracting and removing {zip_path} ...")
    !unzip -q -o {zip_path} -d DIOR_RSVG
    os.remove(zip_path)
!echo "--- after extraction ---"
!find DIOR_RSVG -maxdepth 2 | head -20
!du -sh DIOR_RSVG

## 4. Parse the XML annotations into ODVG grounding JSONL

Mirrors the official `data_loader.py` exactly: for each `<object>` in each XML file,
`member[0]`=category name, `member[2]`=`bndbox` (xmin,ymin,xmax,ymax), `member[3]`=the referring
expression. `train.txt` / `val.txt` / `test.txt` are indices into this flattened
(image, object)-pair list, walked in the same sorted-filename order the original loader uses --
**verified** against the actual source
(github.com/ZhanYang-nwpu/RSVG-pytorch/blob/main/data_loader.py, fetched and compared directly):
same `os.walk` + full-path sort over the Annotations XMLs, same per-file `root.findall('object')`
order, same flat monotonic `count` used as the index the split `.txt` files reference. The only
difference is deliberate and behavior-preserving -- parsing all objects first and filtering by
`index in {train,val,test}_ids` (a set) afterward, instead of the original's single-pass
`if count in Index` (a list) -- identical resulting split membership either way.

One JSONL line per referring expression (not grouped by image) — this matches how DIOR-RSVG,
RSVG-HR, and OPT-RSVG are all trained/evaluated in the literature: one (image, query, box) triplet
per sample.

In [ ]:
import os, json, pickle
import xml.etree.ElementTree as ET
from PIL import Image

DIOR_ROOT = "/kaggle/working/DIOR_RSVG"
ANNO_DIR  = os.path.join(DIOR_ROOT, "Annotations")
IMG_DIR   = os.path.join(DIOR_ROOT, "JPEGImages")

def load_split_ids(split):
    with open(os.path.join(DIOR_ROOT, f"{split}.txt")) as f:
        return set(int(x.strip()) for x in f if x.strip())

def get_image_size(xml_root, image_path):
    w_el, h_el = xml_root.find("./size/width"), xml_root.find("./size/height")
    if w_el is not None and h_el is not None:
        return int(w_el.text), int(h_el.text)
    with Image.open(image_path) as im:
        return im.size  # (width, height)

def parse_all_objects():
    xml_files = sorted(
        os.path.join(dp, f) for dp, _, fs in os.walk(ANNO_DIR) for f in fs if f.endswith(".xml")
    )
    records, count = [], 0
    for xp in xml_files:
        root = ET.parse(xp).getroot()
        filename = root.find("./filename").text
        w, h = get_image_size(root, os.path.join(IMG_DIR, filename))
        for member in root.findall("object"):
            category = member[0].text
            x1, y1, x2, y2 = (float(member[2][0].text), float(member[2][1].text),
                               float(member[2][2].text), float(member[2][3].text))
            expression = member[3].text
            records.append(dict(index=count, filename=filename, category=category,
                                 bbox=[x1, y1, x2, y2], width=w, height=h, expression=expression))
            count += 1
    return records

records = parse_all_objects()
n_images = len({r["filename"] for r in records})
print(f"Parsed {len(records)} (image, expression) pairs across {n_images} images")
categories = sorted({r["category"] for r in records})
print(f"{len(categories)} categories:", categories)

In [ ]:
def to_odvg_line(rec):
    x1, y1, x2, y2 = rec["bbox"]
    return json.dumps({
        "filename": rec["filename"],
        "height": rec["height"],
        "width": rec["width"],
        "grounding": {
            "caption": rec["expression"],
            "regions": [{"bbox": [x1, y1, x2, y2], "phrase": rec["expression"]}],
        },
    })

train_ids, val_ids, test_ids = load_split_ids("train"), load_split_ids("val"), load_split_ids("test")

train_lines = [to_odvg_line(r) for r in records if r["index"] in train_ids]
val_lines   = [to_odvg_line(r) for r in records if r["index"] in val_ids]
test_records = [r for r in records if r["index"] in test_ids]

os.makedirs("/kaggle/working/data", exist_ok=True)
with open("/kaggle/working/data/dior_rsvg_train_grounding.jsonl", "w") as f:
    f.write("\n".join(train_lines))
with open("/kaggle/working/data/dior_rsvg_val_grounding.jsonl", "w") as f:
    f.write("\n".join(val_lines))
with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "wb") as f:
    pickle.dump(test_records, f)

print(f"train={len(train_lines)} (paper: 26991)  val={len(val_lines)} (paper: 3829)  "
      f"test={len(test_records)} (paper: 7500)")

## 4a. Download VRSBench and convert its `[refer]` examples to the same ODVG format

VRSBench (https://huggingface.co/datasets/xiang709/VRSBench, CC-BY-4.0) ships train annotations as
one LLaVA-style conversation file (`VRSBench_train.json`) with `[caption]` / `[refer]` / `[vqa]`
tagged turns -- we only want the `[refer]` ones here. Verified against the actual file before
writing this cell (unchanged from v2):

- Human turn: `[refer] ... <p>{referring expression}</p> ...`
- GPT turn: `{<x1><y1><x2><y2>}`, coordinates **normalized 0-100** per the dataset authors' README
  (https://github.com/lx709/VRSBench) -- *not* 0-1 or absolute pixels.
- ~5.9% of boxes have a coordinate outside [0, 100] (annotation noise, includes some negative
  values) -- clip to [0, 100] rather than discard, then drop any box that's degenerate after
  clipping (x2<=x1 or y2<=y1).

**Train/test leak, found live and confirmed by spot-checking actual image content (not just
matching ids)**: a meaningful slice of VRSBench's images are DIOR's own source images, re-served
under a `<dior_numeric_id>_NNNN.png` filename with independently-written referring expressions --
e.g. DIOR-RSVG's `04691.jpg` ("baseball field, upper left") and VRSBench's `04691_0000.png`
("baseball field... upper left") are the same photo. Left unfiltered, this leaks ~4% of
DIOR-RSVG's own held-out test images back into training via VRSBench, inflating Section 9's "new
checkpoint" numbers. Filtered out below, before the random subset is drawn -- not after, so the
excluded count doesn't quietly shrink `MAX_VRSBENCH_IMAGES`.

Images come from `Images_train.zip` (8.4GB) -- downloaded here, inside Kaggle, not locally.

In [ ]:
%cd /kaggle/working
!wget -q https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/VRSBench_train.json

# Kaggle gives 20GB of working-directory space. Images_train.zip is 8.4GB and the full extraction
# is another ~8GB -- both have to coexist during unzip, which blows the quota before any cleanup
# can run. The [refer] annotations touch almost all of the 20,262 training images, so we cap the
# number of IMAGES instead and extract only those.
#
# MAX_VRSBENCH_IMAGES=4000 -> ~1.6GB extracted (each image averages 0.40MB). Kept at v2's value --
# this run's real budget constraint is training TIME (see the title cell), not disk.
import json, os, random, zipfile

MAX_VRSBENCH_IMAGES = 4000

with open("VRSBench_train.json") as f:
    vrsbench_all = json.load(f)

refer_all = [d for d in vrsbench_all if "[refer]" in d["conversations"][0]["value"]]
all_refer_images = sorted({d["image"] for d in refer_all})

# Exclude anything sharing a numeric id with a DIOR-RSVG TEST-split image (test_records, built in
# Section 4 above) -- confirmed live that VRSBench reuses DIOR's own source images under a renamed
# convention, which would otherwise leak held-out test images back into training.
import re
dior_test_ids = {re.match(r"(\d+)", r["filename"]).group(1) for r in test_records if re.match(r"(\d+)", r["filename"])}
before = len(all_refer_images)
all_refer_images = [
    img for img in all_refer_images
    if not (re.match(r"(\d+)", img) and re.match(r"(\d+)", img).group(1) in dior_test_ids)
]
print(f"Excluded {before - len(all_refer_images)} VRSBench images sharing a numeric id with a "
      f"DIOR-RSVG test-split image ({len(all_refer_images)} remain)")

random.Random(0).shuffle(all_refer_images)
SUBSET_IMAGES = set(all_refer_images[:MAX_VRSBENCH_IMAGES])
print(f"{len(refer_all)} [refer] annotations across {before} images pre-filter; "
      f"keeping {len(SUBSET_IMAGES)} images (test-split-clean)")

!wget -q https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/Images_train.zip
!df -h /kaggle/working | tail -1

VRSBENCH_IMG_DIR = "/kaggle/working/VRSBench_Images_train"
os.makedirs(VRSBENCH_IMG_DIR, exist_ok=True)

extracted = 0
with zipfile.ZipFile("Images_train.zip") as z:
    for info in z.infolist():
        basename = os.path.basename(info.filename)
        if basename in SUBSET_IMAGES:
            info.filename = basename  # flatten -- drop the "Images_train/" prefix
            z.extract(info, VRSBENCH_IMG_DIR)
            extracted += 1
print(f"Extracted {extracted} images")

os.remove("Images_train.zip")  # reclaim 8.4GB immediately -- do NOT leave this until later
!du -sh {VRSBENCH_IMG_DIR}
!df -h /kaggle/working | tail -1

In [ ]:
import json, re
from PIL import Image

p_pattern = re.compile(r"<p>(.*?)</p>")
box_pattern = re.compile(r"\{<(-?\d+)><(-?\d+)><(-?\d+)><(-?\d+)>\}")

# Only the annotations whose image actually got extracted above.
refer_items = [d for d in refer_all if d["image"] in SUBSET_IMAGES]
print(f"{len(refer_items)} [refer] annotations for the {len(SUBSET_IMAGES)} extracted images")

vrsbench_lines = []
skipped_no_image = 0
skipped_degenerate = 0
image_size_cache = {}

for item in refer_items:
    human = item["conversations"][0]["value"]
    gpt = item["conversations"][1]["value"]
    p_match = p_pattern.search(human)
    box_match = box_pattern.search(gpt)
    if not p_match or not box_match:
        continue

    filename = item["image"]
    if filename not in image_size_cache:
        try:
            with Image.open(f"{VRSBENCH_IMG_DIR}/{filename}") as im:
                image_size_cache[filename] = im.size  # (width, height)
        except FileNotFoundError:
            skipped_no_image += 1
            continue
    w, h = image_size_cache[filename]

    # Coordinates are normalized 0-100 per the VRSBench authors' README; ~5.9% fall outside that
    # range (including negatives), so clip rather than discard, then drop anything degenerate.
    x1, y1, x2, y2 = (max(0, min(100, int(v))) for v in box_match.groups())
    if x2 <= x1 or y2 <= y1:
        skipped_degenerate += 1
        continue
    x1, y1, x2, y2 = x1 / 100 * w, y1 / 100 * h, x2 / 100 * w, y2 / 100 * h

    phrase = p_match.group(1)
    vrsbench_lines.append(json.dumps({
        "filename": filename,
        "height": h,
        "width": w,
        "grounding": {"caption": phrase, "regions": [{"bbox": [x1, y1, x2, y2], "phrase": phrase}]},
    }))

print(f"Converted {len(vrsbench_lines)} usable examples "
      f"(skipped {skipped_no_image} missing images, {skipped_degenerate} degenerate boxes)")

with open("/kaggle/working/data/vrsbench_train_grounding.jsonl", "w") as f:
    f.write("\n".join(vrsbench_lines))

## 4b. Add DOTA -- fixed-category detection data, not referring expressions

DIOR-RSVG and VRSBench are both "grounding-style" ODVG data: free-text captions/phrases per box.
DOTA is different -- a fixed 15-16 category aerial object-detection dataset (plane, ship,
storage-tank, harbor, bridge, helicopter, large-vehicle, small-vehicle, roundabout, swimming-pool,
baseball-diamond, tennis-court, basketball-court, ground-track-field, soccer-ball-field,
container-crane). Open-GroundingDino supports this as "detection-style" ODVG -- each region gets a
`label` (int) + `category` (name) instead of a free-text `caption`, with a separate `label_map.json`
(`{"0": "plane", ...}`) the dataset config points at -- **verified against Open-GroundingDino's own
`data_format.md`** before writing the conversion cell below, not guessed or assumed from the
grounding-style format already used above.

**Source**: `kolos1/dota-coco-format` on Kaggle -- add it via **Add Input** (see the markdown cell
in Section 2b above) alongside the checkpoint dataset. Chosen after checking several DOTA mirrors
via the Kaggle API (`kaggle datasets list -s "DOTA aerial"`) for real usability signals (download
counts, usability rating) rather than picking the first result: this one is pre-split into
512x512 patches (DOTA's native images run up to 20,000x20,000px -- unusable directly) with real
COCO-format (axis-aligned) annotations -- **verified directly** by downloading and parsing its
`instances_val.json` before committing to it (2000 images, 29024 boxes, 16 sane category names,
not just trusting the file listing), at 5.4GB total, comfortably inside Kaggle's per-notebook
input limits. Its own page tags the license MIT, but the real constraint is DOTA's upstream
academic/non-commercial license (captain-whu.github.io/DOTA) -- noted here so it isn't
misrepresented by the mirror's own (incorrect) tag.

**`MAX_DOTA_IMAGES`, same reasoning as VRSBench's cap**: this run's binding constraint is training
*time*, not disk or license -- see the title cell's data-budget note. 2000 images (the full `val`
split's image count, already verified above) is a reasonable, bounded addition.

In [ ]:
import glob, json, os, random

# Same reasoning as CURRENT_CKPT_GLOB in Section 2b -- recursive, since an API-pushed kernel
# mounts dataset_sources one level deeper than the web UI's Add Input flow.
dota_matches = glob.glob("/kaggle/input/**/instances_train.json", recursive=True)
assert dota_matches, (
    "No DOTA instances_train.json found under /kaggle/input -- did you add kolos1/dota-coco-format "
    "as an input? See the markdown cell above."
)
DOTA_TRAIN_JSON = dota_matches[0]
DOTA_IMG_DIR = os.path.join(os.path.dirname(os.path.dirname(DOTA_TRAIN_JSON)), "train")
assert os.path.isdir(DOTA_IMG_DIR), f"Expected DOTA train images at {DOTA_IMG_DIR}, not found"
print("DOTA train annotations:", DOTA_TRAIN_JSON)
print("DOTA train images:", DOTA_IMG_DIR)

with open(DOTA_TRAIN_JSON) as f:
    dota_coco = json.load(f)

dota_categories = sorted(dota_coco["categories"], key=lambda c: c["id"])
print(f"{len(dota_coco['images'])} DOTA train images, {len(dota_coco['annotations'])} boxes, "
      f"{len(dota_categories)} categories:", [c["name"] for c in dota_categories])

MAX_DOTA_IMAGES = 2000

all_dota_image_ids = [im["id"] for im in dota_coco["images"]]
random.Random(0).shuffle(all_dota_image_ids)
dota_subset_ids = set(all_dota_image_ids[:MAX_DOTA_IMAGES])
print(f"Keeping {len(dota_subset_ids)} of {len(all_dota_image_ids)} DOTA train images")

# label_map.json: Open-GroundingDino's own doc says indices must start from 0 -- DOTA's own COCO
# category ids already do (checked above), but re-derive rather than assume, in case that ever
# isn't true of a different mirror.
dota_cat_name_by_id = {}
dota_label_map = {}
for new_idx, cat in enumerate(dota_categories):
    dota_label_map[str(new_idx)] = cat["name"]
    dota_cat_name_by_id[cat["id"]] = (new_idx, cat["name"])

os.makedirs("/kaggle/working/data", exist_ok=True)
with open("/kaggle/working/data/dota_label_map.json", "w") as f:
    json.dump(dota_label_map, f, indent=2)
print("Wrote data/dota_label_map.json:", dota_label_map)

In [ ]:
dota_images_by_id = {im["id"]: im for im in dota_coco["images"] if im["id"] in dota_subset_ids}
dota_anns_by_image = {}
for ann in dota_coco["annotations"]:
    if ann["image_id"] in dota_images_by_id:
        dota_anns_by_image.setdefault(ann["image_id"], []).append(ann)

dota_lines = []
skipped_no_annotations = 0
for img_id, im in dota_images_by_id.items():
    anns = dota_anns_by_image.get(img_id, [])
    if not anns:
        skipped_no_annotations += 1
        continue
    instances = []
    for ann in anns:
        x, y, w, h = ann["bbox"]  # COCO: [x, y, width, height] -- axis-aligned, already verified
        label_idx, cat_name = dota_cat_name_by_id[ann["category_id"]]
        instances.append({"bbox": [x, y, x + w, y + h], "label": label_idx, "category": cat_name})
    dota_lines.append(json.dumps({
        "filename": im["file_name"],
        "height": im["height"],
        "width": im["width"],
        "detection": {"instances": instances},
    }))

print(f"Converted {len(dota_lines)} DOTA images to detection-style ODVG "
      f"(skipped {skipped_no_annotations} with no boxes in this subset)")

with open("/kaggle/working/data/dota_train_detection.jsonl", "w") as f:
    f.write("\n".join(dota_lines))

## 5. Build a small COCO-format val set (for Open-GroundingDino's built-in periodic eval)

Their training loop's periodic validation only supports COCO-format detection data (fixed
categories, not free-text queries) — see the README. This is **only a training-time sanity signal**
("is box quality trending the right way"), not the metric that actually matters for your use case.
The real grounding accuracy (Acc@0.5 / Acc@0.7 / mIoU on text queries) is computed separately in
Section 9, after training, on the untouched DIOR-RSVG test split -- DOTA doesn't get its own held-out
number in this notebook (it's training-time-only supervision here, broadening detected categories/
robustness, not something this notebook's eval protocol is set up to score separately).

Capped to a few hundred images to keep the periodic eval fast during a "quick" run — bump
`MAX_VAL_IMAGES` up if you want a more thorough in-training signal.

In [ ]:
MAX_VAL_IMAGES = 500

val_records = [r for r in records if r["index"] in val_ids]
cat2id = {c: i for i, c in enumerate(categories)}

val_by_image = {}
for r in val_records:
    val_by_image.setdefault(r["filename"], []).append(r)
val_image_names = list(val_by_image.keys())[:MAX_VAL_IMAGES]

images, annotations, ann_id = [], [], 0
for img_id, fname in enumerate(val_image_names):
    objs = val_by_image[fname]
    images.append({"id": img_id, "file_name": fname, "height": objs[0]["height"], "width": objs[0]["width"]})
    for r in objs:
        x1, y1, x2, y2 = r["bbox"]
        annotations.append({
            "id": ann_id, "image_id": img_id, "category_id": cat2id[r["category"]],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1), "iscrowd": 0,
        })
        ann_id += 1

coco_val = {
    "images": images,
    "annotations": annotations,
    "categories": [{"id": i, "name": c} for c, i in cat2id.items()],
}
with open("/kaggle/working/data/dior_rsvg_val_coco.json", "w") as f:
    json.dump(coco_val, f)
print(f"COCO val: {len(images)} images, {len(annotations)} boxes, {len(categories)} categories")

## 6. Point Open-GroundingDino at all three data sources

`train` is a *list* -- Open-GroundingDino's ODVG loader natively supports multiple `{root, anno}`
sources trained on together, mixing grounding-style (no `label_map`, free-text `caption`/`phrase`
per line -- DIOR-RSVG and VRSBench) and detection-style (`label_map` set, fixed `category`/`label`
per line -- DOTA) sources in the same list.

In [ ]:
dataset_cfg = {
    "train": [
        {
            "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
            "anno": "/kaggle/working/data/dior_rsvg_train_grounding.jsonl",
            "dataset_mode": "odvg",
        },
        {
            "root": VRSBENCH_IMG_DIR + "/",
            "anno": "/kaggle/working/data/vrsbench_train_grounding.jsonl",
            "dataset_mode": "odvg",
        },
        {
            "root": DOTA_IMG_DIR + "/",
            "anno": "/kaggle/working/data/dota_train_detection.jsonl",
            "label_map": "/kaggle/working/data/dota_label_map.json",
            "dataset_mode": "odvg",
        },
    ],
    "val": [{
        "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
        "anno": "/kaggle/working/data/dior_rsvg_val_coco.json",
        "label_map": None,
        "dataset_mode": "coco",
    }],
}
with open("/kaggle/working/Open-GroundingDino/config/datasets_dior_rsvg_v3.json", "w") as f:
    json.dump(dataset_cfg, f, indent=2)
print("Wrote config/datasets_dior_rsvg_v3.json")
print(f"Training on {len(train_lines)} DIOR-RSVG + {len(vrsbench_lines)} VRSBench + "
      f"{len(dota_lines)} DOTA examples = {len(train_lines) + len(vrsbench_lines) + len(dota_lines)} total")

## 7. Patch `config/cfg_odvg.py`

Per the README, evaluating on a non-COCO custom set needs `use_coco_eval = False` plus a
`label_list` of your class names -- this stays DIOR-RSVG's own category list (unchanged from v1/v2):
the periodic COCO-style eval in Section 5 only ever scores against the DIOR-RSVG val set, so
DOTA's categories don't belong here (they're wired in separately via `dota_label_map.json` in the
dataset config, not this file).

In [ ]:
cfg_path = "/kaggle/working/Open-GroundingDino/config/cfg_odvg.py"
with open(cfg_path) as f:
    cfg_text = f.read()

if "use_coco_eval = True" in cfg_text:
    cfg_text = cfg_text.replace("use_coco_eval = True", "use_coco_eval = False")
    with open(cfg_path, "w") as f:
        f.write(cfg_text)
    print("Set use_coco_eval = False")
else:
    print("WARNING: 'use_coco_eval = True' not found verbatim in cfg_odvg.py — "
          "open the file and set use_coco_eval = False by hand.")

with open(cfg_path, "a") as f:
    f.write(f"\nlabel_list = {categories!r}\n")
print("Appended label_list with", len(categories), "categories")

In [ ]:
# Continued fine-tuning on a constrained Kaggle session -- and this time, THREE data sources
# instead of v2's two. v2 (DIOR-RSVG + VRSBench, epochs=3, save_checkpoint_interval=3) was
# cancelled by Kaggle after consuming ~15 of the account's 30 weekly GPU-hours -- almost
# certainly its session time limit, not a training failure. Rather than guess a fix from its
# log, this run is sized down on purpose:
#
# epochs: 2, not v2's 3 -- adding DOTA on top of DIOR-RSVG+VRSBench makes each epoch itself
# already bigger than v2's; cutting epochs is the lever that directly reduces total wall-clock
# time, which is the actual constraint that got v2 cancelled.
#
# save_checkpoint_interval: 1, not v2's 3 -- with epochs=3 and interval=3, v2 likely only ever
# saved a checkpoint (if any) right at the very end, meaning a mid-run cancellation could have
# left NOTHING usable. Saving after every epoch here means even a repeat cancellation leaves a
# real checkpoint behind.
#
# batch_size=6, lr=2e-5, lr_backbone=2e-6, lr_drop=2: unchanged from v2 -- batch_size=6 was
# already found live to be the largest stable value on a T4 (batch_size=8 OOM'd, climbing
# 9.2GB->10.4GB->11.5GB over 30 steps before failing at 13.46GB used against a real ~14.56GB
# ceiling); lr/lr_backbone are deliberately conservative continued-fine-tuning rates (the repo's
# from-scratch default of 1e-4 would risk washing out the existing fine-tune), not touched here
# since nothing about adding a third dataset changes that reasoning.
cfg_overrides = (
    "\n"
    "epochs = 2\n"
    "lr = 2e-5\n"
    "lr_backbone = 2e-6\n"
    "lr_drop = 2\n"
    "batch_size = 6\n"
    "save_checkpoint_interval = 1\n"
)
with open(cfg_path, "a") as f:
    f.write(cfg_overrides)
print("Applied continued-fine-tuning overrides (epochs=2, lr=2e-5, batch_size=6, save every epoch)")

## 8. Train

Same as v1/v2, except `PRETRAIN_MODEL_PATH` points at your *current* fine-tuned checkpoint
(uploaded in step 2b) instead of the raw Swin-T weights -- Open-GroundingDino's `train_dist.sh`
reads this from the environment (confirmed by reading the actual script).

**Found live (v2)**: `train_dist.sh` has a *second* environment-variable-driven placeholder, the
same shape as `PRETRAIN_MODEL_PATH` but easy to miss since it's set via a trailing `--options
text_encoder_type="$TEXT_ENCODER_TYPE"` flag rather than a normal argument:
`TEXT_ENCODER_TYPE=${TEXT_ENCODER_TYPE:-"/path/to/bert-base-uncased"}`. Left unset, training
starts fine, gets through setup, and only dies ~16 minutes in when the text encoder actually tries
to load from the literal path `/path/to/bert-base-uncased` -- and that `--options` override wins
over `cfg_odvg.py`'s own (correct) `text_encoder_type = "bert-base-uncased"`, so fixing the config
file wouldn't have helped. Set explicitly below to the same `bert-base-uncased` Hub id Section 2
already warmed the cache for.

In [ ]:
%cd /kaggle/working/Open-GroundingDino
import torch
GPU_NUM = max(1, torch.cuda.device_count())
print("Training with", GPU_NUM, "GPU(s)")

In [ ]:
import os
%cd /kaggle/working/Open-GroundingDino
os.environ["PRETRAIN_MODEL_PATH"] = CURRENT_CKPT
os.environ["TEXT_ENCODER_TYPE"] = "bert-base-uncased"
# Found live in v2: batch_size=8 OOM'd on fragmented CUDA memory before hitting the T4's actual
# ceiling (see the batch_size comment in Section 7) -- this is PyTorch's own suggested mitigation
# from the OOM error message itself, not a guess.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("PRETRAIN_MODEL_PATH =", os.environ["PRETRAIN_MODEL_PATH"])
print("TEXT_ENCODER_TYPE =", os.environ["TEXT_ENCODER_TYPE"])
print("PYTORCH_ALLOC_CONF =", os.environ["PYTORCH_ALLOC_CONF"])
!bash train_dist.sh {GPU_NUM} config/cfg_odvg.py config/datasets_dior_rsvg_v3.json ./logs/dior_rsvg_run3

In [ ]:
# Confirm the actual checkpoint filename(s) written -- don't assume it, just look:
!ls -la /kaggle/working/Open-GroundingDino/logs/dior_rsvg_run3

## 9. Grounding accuracy: zero-shot vs. your current checkpoint vs. the new one

Same Acc@0.5 / Acc@0.7 / mIoU protocol as v1/v2's Section 9, evaluated on the same held-out
DIOR-RSVG test split -- three ways, so you can see whether this run actually improved on what you
already had before overwriting `dior_rsvg_finetuned.pth` locally.

In [ ]:
import sys, os, pickle
sys.path.insert(0, "/kaggle/working/Open-GroundingDino")
import torch
from PIL import Image

# groundingdino.util.inference (and the repo's own tools/inference_on_a_image.py) import
# groundingdino.datasets.transforms / groundingdino.models -- found live, via this exact cell
# erroring on a real run: neither submodule exists anymore. The Open-GroundingDino repo has been
# restructured upstream since these notebooks were first written -- datasets/, models/, tools/,
# config/, and most of util/ now live at the REPO ROOT, not nested under groundingdino/, which
# itself only still has a util/ subpackage. main.py (used for training all along, and confirmed
# working -- this is a real trained checkpoint, not a guess) already uses these current paths;
# reimplemented load_model/load_image/predict directly against them instead of relying on
# groundingdino.util.inference, which nothing in the current repo has updated to match.
from util.slconfig import SLConfig
from models.GroundingDINO import build_groundingdino as build_model  # models.build_model itself calls an undefined bare build() -- see notebook's own note
from groundingdino.util.utils import clean_state_dict, get_phrases_from_posmap
import datasets.transforms as T

def load_image(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    transform = T.Compose([
        T.RandomResize([800], max_size=1333),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image, _ = transform(image_pil, None)
    return image_pil, image

def load_model(model_config_path, model_checkpoint_path, device="cuda"):
    args = SLConfig.fromfile(model_config_path)
    args.device = device
    model, _, _ = build_model(args)  # build_groundingdino returns (model, criterion, postprocessors)
    checkpoint = torch.load(model_checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(clean_state_dict(checkpoint["model"]), strict=False)
    model.eval()
    return model.to(device)

def predict(model, image, caption, box_threshold=0.25, text_threshold=0.25, device="cuda"):
    caption = caption.lower().strip()
    if not caption.endswith("."):
        caption += "."
    image = image.to(device)
    with torch.no_grad():
        outputs = model(image[None], captions=[caption])
    logits = outputs["pred_logits"].sigmoid()[0]  # (nq, 256)
    boxes = outputs["pred_boxes"][0]  # (nq, 4)
    filt_mask = logits.max(dim=1)[0] > box_threshold
    logits_filt = logits[filt_mask].cpu()
    boxes_filt = boxes[filt_mask].cpu()
    tokenlizer = model.tokenizer
    tokenized = tokenlizer(caption)
    phrases = [get_phrases_from_posmap(logit > text_threshold, tokenized, tokenlizer) for logit in logits_filt]
    return boxes_filt, logits_filt.max(dim=1)[0], phrases

def box_cxcywh_to_xyxy_abs(box_norm, w, h):
    cx, cy, bw, bh = box_norm
    return [(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate(ckpt_path, cfg_path, test_records, image_root, box_th=0.25, text_th=0.25, limit=None):
    model = load_model(cfg_path, ckpt_path)
    ious = []
    subset = test_records[:limit] if limit else test_records
    for i, r in enumerate(subset):
        img_path = os.path.join(image_root, r["filename"])
        image_source, image = load_image(img_path)
        boxes, logits, phrases = predict(model=model, image=image, caption=r["expression"],
                                          box_threshold=box_th, text_threshold=text_th)
        if len(boxes) == 0:
            ious.append(0.0)
            continue
        best_idx = int(logits.argmax())
        pred_xyxy = box_cxcywh_to_xyxy_abs(boxes[best_idx].tolist(), r["width"], r["height"])
        ious.append(iou_xyxy(pred_xyxy, r["bbox"]))
        if i % 500 == 0:
            print(i, "/", len(subset))
    n = len(ious)
    acc5 = sum(x >= 0.5 for x in ious) / n
    acc7 = sum(x >= 0.7 for x in ious) / n
    miou = sum(ious) / n
    return {"n": n, "Acc@0.5": acc5, "Acc@0.7": acc7, "mIoU": miou}

with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "rb") as f:
    test_records = pickle.load(f)

image_root = "/kaggle/working/DIOR_RSVG/JPEGImages"
cfg_path = "/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py"

print("Zero-shot baseline (pretrained, not fine-tuned) on a 1000-item subset:")
print(evaluate("/kaggle/working/Open-GroundingDino/weights/groundingdino_swint_ogc.pth",
                cfg_path, test_records, image_root, limit=1000))


In [ ]:
print("Your CURRENT checkpoint (before this run) on the same 1000-item subset:")
print(evaluate(CURRENT_CKPT, cfg_path, test_records, image_root, limit=1000))

In [ ]:
# Fill in the actual checkpoint filename from the `ls` output in Section 8 above.
NEW_CKPT = "/kaggle/working/Open-GroundingDino/logs/dior_rsvg_run3/checkpoint_best_regular.pth"

print("NEW checkpoint (this run) on the same 1000-item subset:")
print(evaluate(NEW_CKPT, cfg_path, test_records, image_root, limit=1000))

## 10. Export

Exported as `dior_rsvg_finetuned_v3.pth` (not overwriting `v1`/`v2`) so you can compare all three
locally before deciding whether to actually replace
`models/grounding/checkpoints/dior_rsvg_finetuned.pth` -- only do that once the eval numbers above
actually look better than your current checkpoint's.

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/output_model", exist_ok=True)
shutil.copy(NEW_CKPT, "/kaggle/working/output_model/dior_rsvg_finetuned_v3.pth")
shutil.copy("/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py",
            "/kaggle/working/output_model/GroundingDINO_SwinT_OGC.py")
print("Exported to /kaggle/working/output_model/ -- download from this notebook's Output tab.")

## Next

Download `dior_rsvg_finetuned_v3.pth` from this notebook's Output tab. Compare the three eval
numbers from Section 9 above; if `v3` beats your current checkpoint, replace
`models/grounding/checkpoints/dior_rsvg_finetuned.pth` with it locally (same `GroundingDINO_SwinT_OGC.py`
config either way -- the architecture didn't change). If not, keep your current checkpoint --
DOTA's contribution here is broader category robustness (it trains on aerial object classes
DIOR-RSVG doesn't cover, like `container-crane`), which the DIOR-RSVG-only eval above can't fully
credit; a qualitative check with a few real queries may be more informative than Section 9 alone
for judging whether it actually helped.